# PsychosisRiskCohort — early/uncertain psychosis signal cohort

Per Yize's simplified definition.

**Primary cohort**: age 12-35 inclusive at index date.

**Index date**: first psychosis-risk signal from either:

1. **Psychosis-specific text terms** in any of:
   - `visit_occurrence.visit_source_value`
   - `procedure_occurrence.procedure_source_value`
   - `observation.observation_source_value`
   - `care_site.care_site_name` / `care_site_source_value` (then via visit_occurrence)

   Terms (case-insensitive):
   `psychosis`, `psychotic`, `hallucination`, `delusion`, `paranoia`,
   `schizophrenia`, `schizoaffective`, `prodrome`, `CHR`, `clinical high risk`,
   `SIPS`, `PRIME`, `STEP`, `early psychosis`, `first episode psychosis`, `CMHC`

2. **Early/uncertain psychosis ICD codes** in `condition_occurrence`:
   `F21`, `F23`, `F28`, `F29`

**Kept separate (NOT in this cohort)**: F22, F25.

**Exclusion**: no prior F20-F29 code before index date.

**Excluded broad terms**: `mental`, `behavioral`, `psych`, `psychiatry`, `psychotherapy`
(too non-specific — would capture depression/anxiety/PTSD care, not psychosis risk).

**Outputs**:
1. total patient count
2. counts by signal type
3. top matched `source_value` strings per table
4. age distribution at index
5. later F20-F29 diagnosis within 6 months / 1 year / 2 years post-index
6. patient-level table with person_id, index_date, age_at_index, index_signal_type,
   matched_source_value, later_F20F29 indicators


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import duckdb
import re

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

path = '/home/jupyter/2836994-data2'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

MIN_AGE = 12
MAX_AGE = 35

con = duckdb.connect()
print('DuckDB:', duckdb.__version__)
print('Data path:', path)


## 2. Define psychosis-specific text terms

Each term is matched as a case-insensitive substring. Word-boundary acronyms (CHR, SIPS,
PRIME, STEP, CMHC) are matched with regex word boundaries to avoid false matches inside
longer words (e.g. "STEP" should not match "STEPPED CARE").


In [ ]:
# Substring terms (case-insensitive LIKE %term%)
SUBSTRING_TERMS = [
    'psychosis',
    'psychotic',
    'hallucination',
    'delusion',
    'paranoia',
    'schizophrenia',
    'schizoaffective',
    'prodrome',
    'clinical high risk',
    'early psychosis',
    'first episode psychosis',
]

# Word-boundary acronyms (regex \bACRONYM\b)
ACRONYM_TERMS = ['CHR', 'SIPS', 'PRIME', 'STEP', 'CMHC']

def build_text_match_sql(col):
    """Build a SQL boolean expression that matches if `col` contains any psychosis term."""
    parts = []
    for t in SUBSTRING_TERMS:
        parts.append(f"lower({col}) LIKE '%{t.lower()}%'")
    for a in ACRONYM_TERMS:
        # case-insensitive whole-word match using regex
        parts.append(f"regexp_matches({col}, '(^|[^A-Za-z]){a}([^A-Za-z]|$)')")
    return '(' + ' OR '.join(parts) + ')'

# Sanity preview of the constructed SQL
print(build_text_match_sql('observation_source_value')[:400] + ' ...')


## 3. Get F20-F29 ever-history per person (for exclusion of prior diagnoses)

For each person, find the earliest F20-F29 date in `condition_occurrence`. Persons whose
candidate index_date is on or after their earliest F20-F29 will be excluded.


In [ ]:
f20f29_ever = con.sql(f"""
SELECT
  person_id,
  MIN(CAST(condition_start_datetime AS DATE)) AS first_F20_F29_date,
  -- Also flag whether each F20.x subgroup ever occurs (for later analysis)
  MAX(CASE WHEN condition_source_value LIKE 'F20%' THEN 1 ELSE 0 END) AS ever_F20,
  MAX(CASE WHEN condition_source_value LIKE 'F21%' THEN 1 ELSE 0 END) AS ever_F21,
  MAX(CASE WHEN condition_source_value LIKE 'F22%' THEN 1 ELSE 0 END) AS ever_F22,
  MAX(CASE WHEN condition_source_value LIKE 'F23%' THEN 1 ELSE 0 END) AS ever_F23,
  MAX(CASE WHEN condition_source_value LIKE 'F24%' THEN 1 ELSE 0 END) AS ever_F24,
  MAX(CASE WHEN condition_source_value LIKE 'F25%' THEN 1 ELSE 0 END) AS ever_F25,
  MAX(CASE WHEN condition_source_value LIKE 'F28%' THEN 1 ELSE 0 END) AS ever_F28,
  MAX(CASE WHEN condition_source_value LIKE 'F29%' THEN 1 ELSE 0 END) AS ever_F29
FROM read_parquet('{path}/condition_occurrence/*.parquet')
WHERE condition_source_value LIKE 'F2_%'
  AND condition_start_datetime IS NOT NULL
GROUP BY person_id
""").to_df()
print(f'Persons with any F20-F29 code in history: {len(f20f29_ever):,}')
f20f29_ever.head()


In [ ]:
# Register for later joins
con.register('f20f29_ever_df', f20f29_ever)
con.execute('CREATE OR REPLACE TEMP TABLE f20f29_ever AS SELECT * FROM f20f29_ever_df')


## 4. Collect candidate signal events from each source

For each source we collect: (person_id, event_date, signal_type, matched_source_value).
Later we aggregate to find each person's earliest event ('index_date').


### 4a. ICD codes F21, F23, F28, F29

In [ ]:
icd_signals = con.sql(f"""
SELECT
  person_id,
  CAST(condition_start_datetime AS DATE) AS event_date,
  CASE
    WHEN condition_source_value LIKE 'F21%' THEN 'icd_F21'
    WHEN condition_source_value LIKE 'F23%' THEN 'icd_F23'
    WHEN condition_source_value LIKE 'F28%' THEN 'icd_F28'
    WHEN condition_source_value LIKE 'F29%' THEN 'icd_F29'
  END AS signal_type,
  condition_source_value AS matched_source_value
FROM read_parquet('{path}/condition_occurrence/*.parquet')
WHERE (condition_source_value LIKE 'F21%'
    OR condition_source_value LIKE 'F23%'
    OR condition_source_value LIKE 'F28%'
    OR condition_source_value LIKE 'F29%')
  AND condition_start_datetime IS NOT NULL
""").to_df()
print(f'ICD F21/F23/F28/F29 signal rows: {len(icd_signals):,}')
print(f'  unique persons: {icd_signals["person_id"].nunique():,}')
icd_signals['signal_type'].value_counts()


### 4b. visit_occurrence.visit_source_value text match

In [ ]:
visit_match_sql = build_text_match_sql('visit_source_value')
visit_signals = con.sql(f"""
SELECT
  person_id,
  CAST(visit_start_datetime AS DATE) AS event_date,
  'text_visit' AS signal_type,
  visit_source_value AS matched_source_value
FROM read_parquet('{path}/visit_occurrence/*.parquet')
WHERE visit_source_value IS NOT NULL
  AND visit_start_datetime IS NOT NULL
  AND {visit_match_sql}
""").to_df()
print(f'visit_occurrence text-match rows: {len(visit_signals):,}')
print(f'  unique persons: {visit_signals["person_id"].nunique():,}')


### 4c. procedure_occurrence.procedure_source_value text match

In [ ]:
proc_match_sql = build_text_match_sql('procedure_source_value')
proc_signals = con.sql(f"""
SELECT
  person_id,
  CAST(procedure_datetime AS DATE) AS event_date,
  'text_procedure' AS signal_type,
  procedure_source_value AS matched_source_value
FROM read_parquet('{path}/procedure_occurrence/*.parquet')
WHERE procedure_source_value IS NOT NULL
  AND procedure_datetime IS NOT NULL
  AND {proc_match_sql}
""").to_df()
print(f'procedure_occurrence text-match rows: {len(proc_signals):,}')
print(f'  unique persons: {proc_signals["person_id"].nunique():,}')


### 4d. observation.observation_source_value text match

In [ ]:
obs_match_sql = build_text_match_sql('observation_source_value')
obs_signals = con.sql(f"""
SELECT
  person_id,
  CAST(observation_datetime AS DATE) AS event_date,
  'text_observation' AS signal_type,
  observation_source_value AS matched_source_value
FROM read_parquet('{path}/observation/*.parquet')
WHERE observation_source_value IS NOT NULL
  AND observation_datetime IS NOT NULL
  AND {obs_match_sql}
""").to_df()
print(f'observation text-match rows: {len(obs_signals):,}')
print(f'  unique persons: {obs_signals["person_id"].nunique():,}')


### 4e. care_site psychosis-specific names (via visit_occurrence)

We use the same psychosis-specific term list on care_site_name / care_site_source_value.


In [ ]:
care_match_name = build_text_match_sql('care_site_name')
care_match_src = build_text_match_sql('care_site_source_value')

psych_care_sites = con.sql(f"""
SELECT
  care_site_id,
  care_site_name,
  care_site_source_value
FROM read_parquet('{path}/care_site/*.parquet')
WHERE care_site_id IS NOT NULL
  AND ({care_match_name} OR {care_match_src})
""").to_df()
print(f'Care sites matching psychosis-specific terms: {len(psych_care_sites)}')
display(psych_care_sites.head(20))

if len(psych_care_sites):
    site_id_sql = ','.join(str(int(x)) for x in psych_care_sites['care_site_id'].dropna().tolist())
    site_id_to_name = dict(zip(psych_care_sites['care_site_id'].astype('int64'),
                                psych_care_sites['care_site_name'].fillna('').astype(str)))
    care_signals = con.sql(f"""
    SELECT
      person_id,
      CAST(visit_start_datetime AS DATE) AS event_date,
      'care_site' AS signal_type,
      care_site_id
    FROM read_parquet('{path}/visit_occurrence/*.parquet')
    WHERE care_site_id IN ({site_id_sql})
      AND visit_start_datetime IS NOT NULL
    """).to_df()
    care_signals['matched_source_value'] = care_signals['care_site_id'].map(site_id_to_name)
    care_signals = care_signals[['person_id', 'event_date', 'signal_type', 'matched_source_value']]
else:
    care_signals = pd.DataFrame(columns=['person_id', 'event_date', 'signal_type', 'matched_source_value'])

print(f'care_site visit rows: {len(care_signals):,}')
print(f'  unique persons: {care_signals["person_id"].nunique():,}')


## 5. Union all signals and compute per-person earliest event = candidate index_date

In [ ]:
all_signals = pd.concat(
    [icd_signals, visit_signals, proc_signals, obs_signals, care_signals],
    ignore_index=True
)
all_signals['event_date'] = pd.to_datetime(all_signals['event_date'])
print(f'Total signal rows: {len(all_signals):,}')
print(f'Total unique persons (any signal): {all_signals["person_id"].nunique():,}')
print()
print('Signal rows by type:')
print(all_signals['signal_type'].value_counts())


In [ ]:
# For each person, find the earliest signal event = candidate index_date.
# Record which signal type and source_value produced that earliest event
# (ties broken by signal_type alphabetical order, then by matched_source_value).
candidate_index = (
    all_signals
    .sort_values(['person_id', 'event_date', 'signal_type', 'matched_source_value'])
    .groupby('person_id', as_index=False)
    .first()
    .rename(columns={
        'event_date': 'index_date',
        'signal_type': 'index_signal_type',
    })
)
print(f'Candidate index (one row per person): {len(candidate_index):,}')
candidate_index.head()


## 6. Apply exclusion: drop persons with any F20-F29 code BEFORE index_date

In [ ]:
candidate = candidate_index.merge(
    f20f29_ever[['person_id', 'first_F20_F29_date']], on='person_id', how='left'
)
candidate['first_F20_F29_date'] = pd.to_datetime(candidate['first_F20_F29_date'])
candidate['has_prior_F20_F29'] = (
    candidate['first_F20_F29_date'].notna()
    & (candidate['first_F20_F29_date'] < candidate['index_date'])
)
print(f'Before exclusion: {len(candidate):,}')
print(f'Excluded (prior F20-F29 before index): {candidate["has_prior_F20_F29"].sum():,}')
post_exclusion = candidate[~candidate['has_prior_F20_F29']].copy()
print(f'After exclusion:  {len(post_exclusion):,}')


## 7. Attach birth date, compute age at index, apply age 12-35 filter

In [ ]:
birth = con.sql(f"""
SELECT person_id, CAST(birth_datetime AS DATE) AS birth_date
FROM read_parquet('{path}/person/*.parquet')
""").to_df()
birth['birth_date'] = pd.to_datetime(birth['birth_date'])

post_exclusion = post_exclusion.merge(birth, on='person_id', how='left')
post_exclusion['age_at_index'] = (
    (post_exclusion['index_date'] - post_exclusion['birth_date']).dt.days / 365.25
)

print('Age distribution at index (before age filter):')
print(post_exclusion['age_at_index'].describe())

age_filtered = post_exclusion[
    (post_exclusion['age_at_index'] >= MIN_AGE) &
    (post_exclusion['age_at_index'] <= MAX_AGE)
].copy()
print()
print(f'After age {MIN_AGE}-{MAX_AGE} filter: {len(age_filtered):,}')


## 8. Compute later F20-F29 diagnosis within 6m / 1y / 2y post-index

For each person in the cohort, look up the earliest F20-F29 code in their full history
(`f20f29_ever`). If that earliest code falls within [index_date, index_date + N days], flag.

Note: by exclusion in section 6, the earliest F20-F29 (if it exists) is guaranteed to be
on or after index_date.


In [ ]:
cohort = age_filtered.merge(
    f20f29_ever, on='person_id', how='left', suffixes=('', '_dup')
)
# (first_F20_F29_date already came in from section 6 merge, but f20f29_ever brings the
# ever_F2x flags; drop dup column if any)
if 'first_F20_F29_date_dup' in cohort.columns:
    cohort = cohort.drop(columns=['first_F20_F29_date_dup'])

cohort['days_to_F20_F29'] = (cohort['first_F20_F29_date'] - cohort['index_date']).dt.days

cohort['later_F20F29_6m'] = (
    cohort['days_to_F20_F29'].notna() &
    (cohort['days_to_F20_F29'] >= 0) &
    (cohort['days_to_F20_F29'] <= 182)
).astype(int)
cohort['later_F20F29_1y'] = (
    cohort['days_to_F20_F29'].notna() &
    (cohort['days_to_F20_F29'] >= 0) &
    (cohort['days_to_F20_F29'] <= 365)
).astype(int)
cohort['later_F20F29_2y'] = (
    cohort['days_to_F20_F29'].notna() &
    (cohort['days_to_F20_F29'] >= 0) &
    (cohort['days_to_F20_F29'] <= 730)
).astype(int)

print('Later F20-F29 progression rates:')
print(f'  within 6 months:  {cohort["later_F20F29_6m"].sum():,} '
      f'({100*cohort["later_F20F29_6m"].mean():.1f}%)')
print(f'  within 1 year:    {cohort["later_F20F29_1y"].sum():,} '
      f'({100*cohort["later_F20F29_1y"].mean():.1f}%)')
print(f'  within 2 years:   {cohort["later_F20F29_2y"].sum():,} '
      f'({100*cohort["later_F20F29_2y"].mean():.1f}%)')


## 9. Deliverables

### 9.1  Total number of patients


In [ ]:
print(f'PRIMARY COHORT N = {len(cohort):,}')


### 9.2  Counts by signal type (which signal produced each person's index event)

In [ ]:
signal_type_counts = cohort['index_signal_type'].value_counts().reset_index()
signal_type_counts.columns = ['index_signal_type', 'n_persons']
signal_type_counts['pct'] = (100 * signal_type_counts['n_persons'] / len(cohort)).round(2)
display(signal_type_counts)
signal_type_counts.to_csv(OUTPUT_DIR / 'psychosis_signal_type_counts.csv', index=False)


### 9.3  Top matched source values per source table

Restricted to cohort members (persons whose index event came from each table).
We additionally show, for each table, the top source_values among ALL signal events
in the cohort (not only the index event), since the same person can have multiple events.


In [ ]:
# Restrict the full signal table to cohort members
all_signals_cohort = all_signals[all_signals['person_id'].isin(cohort['person_id'])].copy()

for sig_type in sorted(all_signals_cohort['signal_type'].unique()):
    sub = all_signals_cohort[all_signals_cohort['signal_type'] == sig_type]
    top = (sub.groupby('matched_source_value')
              .agg(n_rows=('person_id', 'size'),
                   n_persons=('person_id', 'nunique'))
              .sort_values('n_persons', ascending=False)
              .head(30)
              .reset_index())
    print(f'\n=== Top matched source_value for signal_type = {sig_type} ===')
    display(top)
    top.to_csv(OUTPUT_DIR / f'psychosis_top_sources_{sig_type}.csv', index=False)


### 9.4  Age distribution at index

In [ ]:
print('Age at index (cohort):')
print(cohort['age_at_index'].describe())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cohort['age_at_index'].dropna(), bins=range(MIN_AGE, MAX_AGE + 2))
ax.set_xlabel('Age at index')
ax.set_ylabel('Persons')
ax.set_title(f'Age distribution at index (n={len(cohort):,})')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'psychosis_age_distribution.png', dpi=120)
plt.show()

age_bins = pd.cut(cohort['age_at_index'],
                  bins=[12, 18, 21, 25, 30, 36],
                  right=False,
                  labels=['12-17', '18-20', '21-24', '25-29', '30-35'])
age_bin_counts = age_bins.value_counts().sort_index().reset_index()
age_bin_counts.columns = ['age_bin', 'n_persons']
display(age_bin_counts)
age_bin_counts.to_csv(OUTPUT_DIR / 'psychosis_age_bins.csv', index=False)


### 9.5  Later F20-F29 progression

In [ ]:
progression = pd.DataFrame([{
    'window': '6 months',
    'n_progressed': int(cohort['later_F20F29_6m'].sum()),
    'pct_progressed': round(100 * cohort['later_F20F29_6m'].mean(), 2),
}, {
    'window': '1 year',
    'n_progressed': int(cohort['later_F20F29_1y'].sum()),
    'pct_progressed': round(100 * cohort['later_F20F29_1y'].mean(), 2),
}, {
    'window': '2 years',
    'n_progressed': int(cohort['later_F20F29_2y'].sum()),
    'pct_progressed': round(100 * cohort['later_F20F29_2y'].mean(), 2),
}])
display(progression)
progression.to_csv(OUTPUT_DIR / 'psychosis_progression.csv', index=False)

# Progression rate broken down by index_signal_type
prog_by_type = (cohort.groupby('index_signal_type')
                .agg(n=('person_id', 'size'),
                     pct_6m=('later_F20F29_6m', lambda x: round(100 * x.mean(), 2)),
                     pct_1y=('later_F20F29_1y', lambda x: round(100 * x.mean(), 2)),
                     pct_2y=('later_F20F29_2y', lambda x: round(100 * x.mean(), 2)))
                .reset_index()
                .sort_values('n', ascending=False))
print('\nProgression rate by index_signal_type:')
display(prog_by_type)
prog_by_type.to_csv(OUTPUT_DIR / 'psychosis_progression_by_signal.csv', index=False)


### 9.6  Patient-level table

Columns: `person_id`, `index_date`, `age_at_index`, `index_signal_type`,
`matched_source_value`, `later_F20F29_6m`, `later_F20F29_1y`, `later_F20F29_2y`,
plus diagnostic provenance flags.


In [ ]:
patient_level = cohort[[
    'person_id', 'index_date', 'age_at_index',
    'index_signal_type', 'matched_source_value',
    'later_F20F29_6m', 'later_F20F29_1y', 'later_F20F29_2y',
    'days_to_F20_F29',
    'ever_F20', 'ever_F21', 'ever_F22', 'ever_F23',
    'ever_F24', 'ever_F25', 'ever_F28', 'ever_F29',
]].copy()

patient_level['age_at_index'] = patient_level['age_at_index'].round(2)

out_pq = OUTPUT_DIR / 'psychosis_risk_cohort.parquet'
out_csv = OUTPUT_DIR / 'psychosis_risk_cohort.csv'
patient_level.to_parquet(out_pq, index=False)
patient_level.to_csv(out_csv, index=False)
print(f'Saved patient-level table: {len(patient_level):,} rows')
print(f'  parquet: {out_pq}')
print(f'  csv:     {out_csv}')
display(patient_level.head(10))


## 10. Output manifest

In [ ]:
for p in sorted(OUTPUT_DIR.glob('psychosis_*')):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:50s}  {size_kb:8.1f} KB')
